# 16 - The chosen guided condition on the test sample

Notebook 15 chose K1_guide_shots on the development sample (mean AUC-ROC 0.688 against 0.681 for K1_guide). The test sample (255 accounts, the same strata) is read once in it and compared with K1 on the same accounts. This shows whether the gain holds on accounts that played no part in writing the guide or choosing the condition.

Plan fixed on 15 November 2026, before any test answer was read: the same guide, examples and reference accounts as on the development sample; measures and rule as in notebooks 14 and 15 (seed 15); no other condition and no changed prompt is run on the test sample.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import binomtest, norm
from sklearn.metrics import roc_auc_score

from skeptical_second_reader import guide
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.pipeline import READ_ALONE, READ_SHOTS
from skeptical_second_reader.reading_sample import (
    MATCHED_NAME,
    OUTSIDE_NAME,
    RANDOM_NAME,
)
from skeptical_second_reader.second_reader import NUM_CTX, SUSPICIOUS

MODELS = ["gemma4:latest", "qwen3.5:9b-q4_K_M"]
CHOSEN = READ_SHOTS
CONDITIONS = [READ_ALONE, CHOSEN]
CLEAN = [MATCHED_NAME, RANDOM_NAME]
DRAWS = 2000
SEED = 15

reading_dir = DATA_INTERIM / "reading"
samples = {
    "dev": pd.read_parquet(DATA_INTERIM / "reading_dev_cases.parquet")["stratum"],
    "test": pd.read_parquet(DATA_INTERIM / "reading_test_cases.parquet")["stratum"],
}


def load(sample):
    alone = pd.read_parquet(reading_dir / f"results_{sample}.parquet")
    guided = pd.read_parquet(reading_dir / f"results_{sample}_guided.parquet")
    out = pd.concat(
        [
            alone[alone["condition"] == READ_ALONE],
            guided[guided["condition"] == CHOSEN],
        ],
        ignore_index=True,
    )
    out["usable"] = out["decision"].notna() & out["confidence"].notna()
    out["says"] = out["decision"] == SUSPICIOUS
    out["conf_score"] = np.where(out["says"], out["confidence"], 1 - out["confidence"])
    return out


results = {name: load(name) for name in samples}
test = samples["test"]
prompts = {
    name: pd.read_parquet(reading_dir / f"prompts_{name}_guided.parquet")
    for name in samples
}
ceilings = pd.read_parquet(DATA_INTERIM / "ceiling_facts_scores.parquet")
truth = {name: (~s.isin(CLEAN)).astype(int) for name, s in samples.items()}
patterns = sorted(s for s in test.unique() if s.startswith("pattern "))
ORDER = patterns + [OUTSIDE_NAME] + CLEAN
assert test.index.intersection(samples["dev"].index).empty
print(
    f"test accounts {len(test)}  positives {truth['test'].sum()}  "
    f"clean {(truth['test'] == 0).sum()}"
)
print(test.value_counts().reindex(ORDER).to_string())

test accounts 255  positives 135  clean 120
stratum
pattern BIPARTITE             15
pattern CYCLE                 15
pattern FAN-IN                15
pattern FAN-OUT               15
pattern GATHER-SCATTER        15
pattern RANDOM                15
pattern SCATTER-GATHER        15
pattern STACK                 15
positive, outside the file    15
clean, matched activity       60
clean, random                 60


## 1. Integrity

Checks that every test account was read once per condition with the model versions of the development sample, that the test prompts carry the same guide and examples, how many answers are not usable, and that no prompt was cut (4,096 tokens).

In [2]:
r, d = results["test"], results["dev"]
assert set(r["model"]) == set(MODELS)
assert set(r["condition"]) == set(CONDITIONS)
assert not r.duplicated(["model", "condition", "account"]).any()
for (model, condition), part in r.groupby(["model", "condition"]):
    assert set(part["account"]) == set(test.index), (model, condition)
assert (r["group"] == test.loc[r["account"]].to_numpy()).all()
assert (r.groupby("model")["digest"].nunique() == 1).all()
assert (
    r.groupby("model")["digest"].first() == d.groupby("model")["digest"].first()
).all()
assert r["think"].nunique() == 1 and set(r["think"].astype(str)) == set(
    d["think"].astype(str)
)


def shots_part(prompt):
    start = prompt.index("SOLVED EXAMPLES")
    return prompt[start : prompt.index("\n\nACCOUNT A", start)]


chosen_prompts = {
    name: p.loc[p["condition"] == CHOSEN, "prompt"] for name, p in prompts.items()
}
assert len(chosen_prompts["test"]) == len(test)
for name, texts in chosen_prompts.items():
    assert texts.str.contains(guide.GUIDE, regex=False).all(), name
blocks = {name: set(texts.map(shots_part)) for name, texts in chosen_prompts.items()}
assert len(blocks["dev"]) == 1 and blocks["test"] == blocks["dev"]
print("guide and examples: the same in every development and test prompt")

r["total_tokens"] = r["prompt_tokens"] + r["answer_tokens"]
tokens = r.groupby(["model", "condition"]).agg(
    answers=("account", "size"),
    not_usable=("usable", lambda s: int((~s).sum())),
    score_missing=("score", lambda s: int(s.isna().sum())),
    max_prompt_tokens=("prompt_tokens", "max"),
    max_prompt_and_answer=("total_tokens", "max"),
    at_window=("prompt_tokens", lambda s: int((s >= NUM_CTX).sum())),
)
print(f"context window {NUM_CTX}")
print(tokens.to_string())

guide and examples: the same in every development and test prompt


context window 4096
                                  answers  not_usable  score_missing  max_prompt_tokens  max_prompt_and_answer  at_window
model             condition                                                                                              
gemma4:latest     K1                  255           0            255               1547                   1769          0
                  K1_guide_shots      255           0              0               3025                   3179          0
qwen3.5:9b-q4_K_M K1                  255           0            255               1511                   1701          0
                  K1_guide_shots      255           0              0               2994                   3130          0


## 2. Share flagged per stratum

How often each reader says suspicious in each group, with Wilson 95% intervals: recall on the positives, false alarms on the clean accounts. Shows where the guide changes the decisions.

In [3]:
def wilson(k, n, level=0.95):
    if n == 0:
        return np.nan, np.nan
    z = norm.ppf(0.5 + level / 2)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


def usable_accounts(frame):
    ok = frame.groupby(["model", "account"])["usable"].all()
    return {m: ok[m][ok[m]].index for m in MODELS}


paired = usable_accounts(r)
use = r[[a in paired[m] for m, a in zip(r["model"], r["account"], strict=True)]].copy()
for m in MODELS:
    print(f"{m}: accounts usable in both conditions {len(paired[m])}")

rows = []
for (model, condition, stratum), part in use.groupby(["model", "condition", "group"]):
    k, n = int(part["says"].sum()), len(part)
    low, high = wilson(k, n)
    rows.append([model, condition, stratum, k, n, k / n, low, high])
flagged = pd.DataFrame(
    rows,
    columns=["model", "condition", "stratum", "flagged", "n", "share", "low", "high"],
)
for model in MODELS:
    mine = flagged[flagged["model"] == model].copy()
    mine["text"] = mine.apply(
        lambda r: (
            f"{r['flagged']:>2}/{r['n']:<2} {r['share']:.2f} "
            f"[{r['low']:.2f}, {r['high']:.2f}]"
        ),
        axis=1,
    )
    print(model)
    print(
        mine.pivot(index="stratum", columns="condition", values="text")
        .reindex(ORDER)[CONDITIONS]
        .to_string()
    )
    print()

gemma4:latest: accounts usable in both conditions 255
qwen3.5:9b-q4_K_M: accounts usable in both conditions 255
gemma4:latest
condition                                        K1           K1_guide_shots
stratum                                                                     
pattern BIPARTITE            8/15 0.53 [0.30, 0.75]   4/15 0.27 [0.11, 0.52]
pattern CYCLE                8/15 0.53 [0.30, 0.75]   8/15 0.53 [0.30, 0.75]
pattern FAN-IN              10/15 0.67 [0.42, 0.85]   6/15 0.40 [0.20, 0.64]
pattern FAN-OUT              6/15 0.40 [0.20, 0.64]   1/15 0.07 [0.01, 0.30]
pattern GATHER-SCATTER       7/15 0.47 [0.25, 0.70]   3/15 0.20 [0.07, 0.45]
pattern RANDOM              12/15 0.80 [0.55, 0.93]  11/15 0.73 [0.48, 0.89]
pattern SCATTER-GATHER      11/15 0.73 [0.48, 0.89]   7/15 0.47 [0.25, 0.70]
pattern STACK               12/15 0.80 [0.55, 0.93]   6/15 0.40 [0.20, 0.64]
positive, outside the file   6/15 0.40 [0.20, 0.64]   0/15 0.00 [0.00, 0.20]
clean, matched activity    

## 3. Paired comparison against K1

The main result: the AUC-ROC change from K1 to the guided reading on the same accounts, with a 95% interval, and how many decisions became right or wrong (McNemar, positives and clean apart).

In [4]:
def draws_for(strata, seed):
    rng = np.random.default_rng(seed)
    index = [np.flatnonzero(strata == s) for s in ORDER]
    return [
        np.concatenate(
            [rng.choice(ix, size=len(ix), replace=True) for ix in index if len(ix)]
        )
        for _ in range(DRAWS)
    ]


def paired_change(y, base, new, draws):
    a, b = roc_auc_score(y, base), roc_auc_score(y, new)
    boot = np.array(
        [roc_auc_score(y[d], new[d]) - roc_auc_score(y[d], base[d]) for d in draws]
    )
    low, high = np.percentile(boot, [2.5, 97.5])
    rule = "helped" if low > 0 else ("hurt" if high < 0 else "a help was not shown")
    return a, b, b - a, low, high, rule


def wide_for(frame, model, accounts):
    return (
        frame[frame["model"] == model]
        .pivot(
            index="account", columns="condition", values=["says", "conf_score", "score"]
        )
        .loc[accounts]
    )


mcnemar_rows, auc_rows, wides = [], [], {}
for model in MODELS:
    accounts = paired[model]
    y = truth["test"].loc[accounts].to_numpy()
    draws = draws_for(test.loc[accounts].to_numpy(), SEED)
    wide = wide_for(use, model, accounts)
    wides[model] = (wide, y, draws)
    right = {
        c: wide[("says", c)].astype(bool).to_numpy() == (y == 1) for c in CONDITIONS
    }
    for side, mask in [("positives", y == 1), ("clean", y == 0)]:
        better = int((~right[READ_ALONE] & right[CHOSEN] & mask).sum())
        worse = int((right[READ_ALONE] & ~right[CHOSEN] & mask).sum())
        p = binomtest(better, better + worse, 0.5).pvalue if better + worse else np.nan
        mcnemar_rows.append(
            [
                model,
                side,
                int(mask.sum()),
                int((right[READ_ALONE] & mask).sum()),
                int((right[CHOSEN] & mask).sum()),
                better,
                worse,
                p,
            ]
        )
    base = wide[("conf_score", READ_ALONE)].to_numpy(float)
    new = wide[("conf_score", CHOSEN)].to_numpy(float)
    auc_rows.append([model, *paired_change(y, base, new, draws)])

mcnemar = pd.DataFrame(
    mcnemar_rows,
    columns=[
        "model",
        "accounts",
        "n",
        "right K1",
        "right guided",
        "became right",
        "became wrong",
        "exact p",
    ],
)
auc = pd.DataFrame(
    auc_rows,
    columns=["model", "AUC K1", "AUC guided", "change", "low 95%", "high 95%", "rule"],
)
print(f"McNemar, decisions that change against K1 ({CHOSEN})")
print(mcnemar.round(4).to_string(index=False))
print()
print(f"AUC-ROC (confidence score), stratified paired bootstrap {DRAWS}")
print(auc.round(3).to_string(index=False))

McNemar, decisions that change against K1 (K1_guide_shots)
            model  accounts   n  right K1  right guided  became right  became wrong  exact p
    gemma4:latest positives 135        80            46             8            42      0.0
    gemma4:latest     clean 120        79           119            40             0      0.0
qwen3.5:9b-q4_K_M positives 135        95            61             8            42      0.0
qwen3.5:9b-q4_K_M     clean 120        49           119            70             0      0.0

AUC-ROC (confidence score), stratified paired bootstrap 2000
            model  AUC K1  AUC guided  change  low 95%  high 95%   rule
    gemma4:latest   0.637       0.755   0.118    0.061     0.182 helped
qwen3.5:9b-q4_K_M   0.540       0.737   0.197    0.127     0.269 helped


## 4. Secondary measures

AUC of the decision alone, AUC of the 0-100 score, and tied scores between positive and clean accounts: whether the gain is in the decisions or only in the ranking.

In [5]:
def tie_share(y, s):
    pos, neg = s[y == 1], s[y == 0]
    return float((pos[:, None] == neg[None, :]).mean())


rows = []
for model in MODELS:
    wide, y, draws = wides[model]
    for condition in CONDITIONS:
        says = wide[("says", condition)].astype(float).to_numpy()
        conf = wide[("conf_score", condition)].to_numpy(float)
        row = {
            "model": model,
            "condition": condition,
            "AUC confidence": roc_auc_score(y, conf),
            "AUC decision only": roc_auc_score(y, says),
            "distinct confidence values": len(np.unique(conf)),
            "tied pairs, confidence": tie_share(y, conf),
        }
        if condition == CHOSEN:
            score = wide[("score", condition)].to_numpy(float)
            ok = ~np.isnan(score)
            row["score answers"] = int(ok.sum())
            row["AUC 0-100 score"] = roc_auc_score(y[ok], score[ok])
            row["tied pairs, 0-100 score"] = tie_share(y[ok], score[ok])
        rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))

print()
print("0-100 score change against the K1 confidence score (descriptive, same draws)")
for model in MODELS:
    wide, y, draws = wides[model]
    score = wide[("score", CHOSEN)].to_numpy(float)
    if np.isnan(score).any():
        missing = int(np.isnan(score).sum())
        print(f"  {model}: {missing} answers without a score, skipped")
        continue
    base = wide[("conf_score", READ_ALONE)].to_numpy(float)
    a, b, change, low, high, _ = paired_change(y, base, score, draws)
    print(f"  {model}: {a:.3f} -> {b:.3f} ({change:+.3f} [{low:+.3f}; {high:+.3f}])")

            model      condition  AUC confidence  AUC decision only  distinct confidence values  tied pairs, confidence  score answers  AUC 0-100 score  tied pairs, 0-100 score
    gemma4:latest             K1           0.637              0.625                           6                   0.240            NaN              NaN                      NaN
    gemma4:latest K1_guide_shots           0.755              0.666                           6                   0.297          255.0             0.76                    0.293
qwen3.5:9b-q4_K_M             K1           0.540              0.556                           6                   0.398            NaN              NaN                      NaN
qwen3.5:9b-q4_K_M K1_guide_shots           0.737              0.722                           7                   0.426          255.0             0.75                    0.387

0-100 score change against the K1 confidence score (descriptive, same draws)


  gemma4:latest: 0.637 -> 0.760 (+0.123 [+0.067; +0.186])


  qwen3.5:9b-q4_K_M: 0.540 -> 0.750 (+0.209 [+0.139; +0.281])


## 5. Development and test side by side

The same measures on both samples, to see whether the change found on the development sample holds.

In [6]:
rows = []
for name, frame in results.items():
    ok = usable_accounts(frame)
    for model in MODELS:
        accounts = ok[model]
        y = truth[name].loc[accounts].to_numpy()
        wide = wide_for(frame, model, accounts)
        for condition in CONDITIONS:
            says = wide[("says", condition)].astype(bool).to_numpy()
            rows.append(
                {
                    "sample": name,
                    "model": model.split(":")[0],
                    "condition": condition,
                    "accounts": len(accounts),
                    "AUC confidence": roc_auc_score(
                        y, wide[("conf_score", condition)].to_numpy(float)
                    ),
                    "recall (all positives)": says[y == 1].mean(),
                    "false alarm rate": says[y == 0].mean(),
                }
            )
side = pd.DataFrame(rows)
print(side.round(3).to_string(index=False))

sample   model      condition  accounts  AUC confidence  recall (all positives)  false alarm rate
   dev  gemma4             K1       255           0.597                   0.600             0.433
   dev  gemma4 K1_guide_shots       255           0.692                   0.237             0.042
   dev qwen3.5             K1       255           0.536                   0.733             0.625
   dev qwen3.5 K1_guide_shots       255           0.685                   0.348             0.025
  test  gemma4             K1       255           0.637                   0.593             0.342
  test  gemma4 K1_guide_shots       255           0.755                   0.341             0.008
  test qwen3.5             K1       255           0.540                   0.704             0.592
  test qwen3.5 K1_guide_shots       255           0.737                   0.452             0.008


## 6. Beside the ceiling

The classifiers trained on the prompt evidence and on the evidence and facts, on the same test accounts: what the same information gives a trained model.

In [7]:
share = {}
for name in ["prompt evidence", "evidence and facts"]:
    share[f"ceiling {name}"] = (
        (ceilings.loc[test.index, f"flags {name}"] == 1).groupby(test).mean()
    )
for model in MODELS:
    for condition in CONDITIONS:
        part = use[(use["model"] == model) & (use["condition"] == condition)]
        share[f"{model.split(':')[0]} {condition}"] = part.groupby("group")[
            "says"
        ].mean()
print("share flagged per stratum")
print(pd.DataFrame(share).reindex(ORDER).round(2).to_string())
print()
y_all = truth["test"].loc[test.index].to_numpy()
aucs = {
    f"ceiling {n}": roc_auc_score(y_all, ceilings.loc[test.index, f"scores {n}"])
    for n in ["prompt evidence", "evidence and facts"]
}
for row in auc_rows:
    aucs[f"{row[0].split(':')[0]} {READ_ALONE}"] = row[1]
    aucs[f"{row[0].split(':')[0]} {CHOSEN}"] = row[2]
print("AUC-ROC (readers: accounts usable for that model)")
print(pd.Series(aucs).round(3).to_string())

share flagged per stratum
                            ceiling prompt evidence  ceiling evidence and facts  gemma4 K1  gemma4 K1_guide_shots  qwen3.5 K1  qwen3.5 K1_guide_shots
pattern BIPARTITE                              0.53                        0.47       0.53                   0.27        0.73                    0.40
pattern CYCLE                                  0.67                        0.67       0.53                   0.53        0.80                    0.73
pattern FAN-IN                                 0.53                        0.47       0.67                   0.40        0.80                    0.47
pattern FAN-OUT                                0.13                        0.40       0.40                   0.07        0.47                    0.07
pattern GATHER-SCATTER                         0.33                        0.40       0.47                   0.20        0.47                    0.27
pattern RANDOM                                 0.93                       

## 7. Results

- K1_guide_shots helped both models: AUC-ROC Gemma 0.637 to 0.755 (+0.118 [+0.061; +0.182]), Qwen 0.540 to 0.737 (+0.197 [+0.127; +0.269]), larger than on the development sample (+0.095, +0.149).
- The decisions moved towards `not suspicious`: 1 false alarm of 120 clean accounts per model (K1: 41 and 71); recall 0.59 to 0.34 (Gemma), 0.70 to 0.45 (Qwen). Of the positives, 42 became wrong and 8 right per model; no clean account became wrong.
- No positive outside the pattern file is flagged (0 of 15).
- The guided readers stay far below the classifiers on the same evidence (AUC-ROC 0.891 and 0.901).
- The guide's strongest sign is probably a mark of the simulator (notebook 15), so the gain may not carry over to other data.